In [ ]:
import pandas as pd
import numpy as np
import pyreadstat as prs
import os
from pathlib import Path
import sys

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'
os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))

# import local toolkit (try normal import first, fall back to loading from file)
try:
	import similarity_toolkit_LLM as stk
except Exception:
	import importlib.util
	toolkit_path = repo_root / 'Code' / 'tools' / 'similarity_toolkit_LLM.py'
	if toolkit_path.exists():
		spec = importlib.util.spec_from_file_location("similarity_toolkit_LLM", str(toolkit_path))
		stk = importlib.util.module_from_spec(spec)
		spec.loader.exec_module(stk)
	else:
		raise

In [ ]:
import re

v_lower = np.vectorize(lambda x: x.lower() if isinstance(x, str) else x)
v_compare = np.vectorize(lambda x, y: x == y)
v_contains = np.vectorize(lambda x, y: y in x if isinstance(x, str) and isinstance(y, str) else False)

# Helper function to remove the code of the question from a string
# A code is a number or a code with a letter and number ended with a dot, e.g. "1.", "2a.", "10b.", "M10."
# If the first component of the string is a word, then do not remove anything.
def remove_question_code(s):
    if not isinstance(s, str):
        return s
    parts = s.split()
    if len(parts) == 0:
        return s
    first_part = parts[0]
    # Match patterns like "1.", "10b.", "m10.", "M10."
    if re.fullmatch(r'[A-Za-z]?\d+[A-Za-z]?\.', first_part):
        return ' '.join(parts[1:])
    else:
        return s
    
# Present the functionality of remove_question_code
print(remove_question_code("1. What is your age?"))  # Should return "What is your age?"
print(remove_question_code("m10. How many people live in your household?"))  # Should return "How many people live in your household?"
print(remove_question_code("How are you?"))  # Should return "How are you?"
print(remove_question_code(123))

# Vectorize the function
v_remove_question_code = np.vectorize(remove_question_code)

# Helper function to find a sequence of characters in a string
# For example, to find "doch" in "Ile wynoszą PRZECIĘTNE MIESIĘCZNE DOCHODY NETTO (NA RĘKĘ) PRZYPADAJĄCE NA JEDNĄ OSOBĘ W PANA(I) GOSPODARSTWIE DOMOWYM?"
def contains_sequence(s, seq):
    if not isinstance(s, str) or not isinstance(seq, str):
        return False
    return seq.lower() in s.lower()

# Present the functionality of contains_sequence
print(contains_sequence("Ile wynoszą PRZECIĘTNE MIESIĘCZNE DOCHODY NETTO (NA RĘKĘ) PRZYPADAJĄCE NA JEDNĄ OSOBĘ W PANA(I) GOSPODARSTWIE DOMOWYM?", "doch"))  # Should return True
print(contains_sequence("How are you?", "ar"))  # Should return True
print(contains_sequence("How are you?", "ae"))  # Should return False
print(contains_sequence("Hello world!", "test"))  # Should return False
print(contains_sequence(123, "test"))  # Should return False

In [ ]:
path_spss = Path("/Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/CBOS SPSS")
list_dir = os.listdir(path_spss)
list_dir = [file for file in list_dir if file.endswith(".sav")]

# Sort by CBOS_XXX where XXX is a number with 1 to 3 digits
list_dir.sort(key=lambda x: int(x.split('_')[1].split('.')[0]))

In [ ]:
main_df = pd.DataFrame()
for file in list_dir:
    file_path = path_spss / file
    df, meta = prs.read_sav(file_path)
    df['survey_year'] = meta.file_header['year']
    df['survey_month'] = meta.file_header['month']
    main_df = pd.concat([main_df, df], ignore_index=True)